In [25]:
import pandas as pd
import numpy as np

In [26]:
df = pd.read_csv("../Data/weather_daily_2013_2023.csv")
df

,Date,District,RF,T max,T min,Avg_tem
0,10/1/2013,Colombo,0.0,31.1,25.8,28.45
1,10/2/2013,Colombo,0.0,31.3,27.5,29.40
2,10/3/2013,Colombo,1.6,31.3,27.3,29.30
3,10/4/2013,Colombo,5.8,30.7,25.7,28.20
4,10/5/2013,Colombo,0.3,30.8,24.4,27.60
...,...,...,...,...,...,...
3647,9/26/2023,Colombo,18.4,31.3,26.4,28.85
3648,9/27/2023,Colombo,42.4,29.8,25.2,27.50
3649,9/28/2023,Colombo,63.4,28.7,24.6,26.65
3650,9/29/2023,Colombo,60.3,29.8,23.8,26.80


In [27]:
df = df.rename(columns={
    'Date': 'date',
    'District': 'district',
    'RF': 'rf',
    'T max ': 't_max',
    'T min': 't_min',
    'Avg_tem': 'avg_temp'
})
print(df.columns)

Index(['date', 'district', 'rf', 't_max', 't_min', 'avg_temp'], dtype='object')


In [28]:
# def classify_weather(row):
#     r = row['rf']
# 
#     if r < 4.0:
#         return 'Very Dry'
#     elif r < 6.6:
#         return 'Dry'
#     elif r <= 8.3:
#         return 'Normal'
#     elif r <= 10.0:
#         return 'Wet'
#     else:
#         return 'Very Wet'


def classify_weather(row):
    r = row['rf']
    if r < 5.0:
        return 'Dry'
    elif r <= 9.0:
        return 'Normal'
    else:
        return 'Wet'


# Apply the function to your DataFrame
df['weather_condition'] = df.apply(classify_weather, axis=1)


In [29]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')

Weather is highly seasonal, especially rainfall.
You can encode the time of year to help the model learn monsoon or dry periods.

In [30]:
df['month'] = df['date'].dt.month
df['dayofyear'] = df['date'].dt.dayofyear
df['sin_day'] = np.sin(2 * np.pi * df['dayofyear'] / 365)
df['cos_day'] = np.cos(2 * np.pi * df['dayofyear'] / 365)
df

,date,district,rf,t_max,t_min,avg_temp,weather_condition,month,dayofyear,sin_day,cos_day
0,2013-10-01,Colombo,0.0,31.1,25.8,28.45,Dry,10,274,-0.999991,0.004304
1,2013-10-02,Colombo,0.0,31.3,27.5,29.40,Dry,10,275,-0.999769,0.021516
2,2013-10-03,Colombo,1.6,31.3,27.3,29.30,Dry,10,276,-0.999250,0.038722
3,2013-10-04,Colombo,5.8,30.7,25.7,28.20,Normal,10,277,-0.998435,0.055917
4,2013-10-05,Colombo,0.3,30.8,24.4,27.60,Dry,10,278,-0.997325,0.073095
...,...,...,...,...,...,...,...,...,...,...,...
3647,2023-09-26,Colombo,18.4,31.3,26.4,28.85,Wet,9,269,-0.996659,-0.081676
3648,2023-09-27,Colombo,42.4,29.8,25.2,27.50,Wet,9,270,-0.997917,-0.064508
3649,2023-09-28,Colombo,63.4,28.7,24.6,26.65,Wet,9,271,-0.998880,-0.047321
3650,2023-09-29,Colombo,60.3,29.8,23.8,26.80,Wet,9,272,-0.999546,-0.030120


create relationships between t_max, t_min, and avg_temp:

In [31]:
# df['temp_range'] = df['t_max'] - df['t_min']          # daytime temp spread
# df['temp_anomaly'] = df['avg_temp'] - df['avg_temp'].mean()  # deviation from mean

Nonlinear or Interaction Features

Some models benefit when you explicitly show how variables interact.

In [32]:
# df['rf_temp_ratio'] = df['rf'] / (df['avg_temp'] + 1)
# df['rf_tmax_product'] = df['rf'] * df['t_max']
# df['tmax_tmin_ratio'] = df['t_max'] / (df['t_min'] + 1)

In [33]:
df

,date,district,rf,t_max,t_min,avg_temp,weather_condition,month,dayofyear,sin_day,cos_day
0,2013-10-01,Colombo,0.0,31.1,25.8,28.45,Dry,10,274,-0.999991,0.004304
1,2013-10-02,Colombo,0.0,31.3,27.5,29.40,Dry,10,275,-0.999769,0.021516
2,2013-10-03,Colombo,1.6,31.3,27.3,29.30,Dry,10,276,-0.999250,0.038722
3,2013-10-04,Colombo,5.8,30.7,25.7,28.20,Normal,10,277,-0.998435,0.055917
4,2013-10-05,Colombo,0.3,30.8,24.4,27.60,Dry,10,278,-0.997325,0.073095
...,...,...,...,...,...,...,...,...,...,...,...
3647,2023-09-26,Colombo,18.4,31.3,26.4,28.85,Wet,9,269,-0.996659,-0.081676
3648,2023-09-27,Colombo,42.4,29.8,25.2,27.50,Wet,9,270,-0.997917,-0.064508
3649,2023-09-28,Colombo,63.4,28.7,24.6,26.65,Wet,9,271,-0.998880,-0.047321
3650,2023-09-29,Colombo,60.3,29.8,23.8,26.80,Wet,9,272,-0.999546,-0.030120


In [34]:
df.columns

Index(['date', 'district', 'rf', 't_max', 't_min', 'avg_temp',
       'weather_condition', 'month', 'dayofyear', 'sin_day', 'cos_day'],
      dtype='object')

In [35]:
df = df.drop(columns=['date', 'district', 'dayofyear'])
df

,rf,t_max,t_min,avg_temp,weather_condition,month,sin_day,cos_day
0,0.0,31.1,25.8,28.45,Dry,10,-0.999991,0.004304
1,0.0,31.3,27.5,29.40,Dry,10,-0.999769,0.021516
2,1.6,31.3,27.3,29.30,Dry,10,-0.999250,0.038722
3,5.8,30.7,25.7,28.20,Normal,10,-0.998435,0.055917
4,0.3,30.8,24.4,27.60,Dry,10,-0.997325,0.073095
...,...,...,...,...,...,...,...,...
3647,18.4,31.3,26.4,28.85,Wet,9,-0.996659,-0.081676
3648,42.4,29.8,25.2,27.50,Wet,9,-0.997917,-0.064508
3649,63.4,28.7,24.6,26.65,Wet,9,-0.998880,-0.047321
3650,60.3,29.8,23.8,26.80,Wet,9,-0.999546,-0.030120


In [36]:
path = "../Data/weather_daily_2013_2023_after_preprocess.csv"

# Save DataFrame to CSV
df.to_csv(path, index=False)

print(f"CSV file saved successfully at: {path}")

CSV file saved successfully at: ../Data/weather_daily_2013_2023_after_preprocess.csv
